# Word2Vec: CBOW, Skip-gram, Average Word2Vec

Lectures 21–27. Uses `gensim`.

- **CBOW**: context words → predict target word
- **Skip-gram**: target word → predict context words (better for rare words)
- **Average Word2Vec**: mean of a document's word vectors → document vector for classical ML

## Train a small Word2Vec model from scratch

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec

text = """Machine learning is transforming the world. Deep learning uses neural networks.
Natural language processing helps computers understand human language.
Word embeddings capture semantic meaning of words. Neural networks learn representations."""

sentences = [word_tokenize(s.lower()) for s in sent_tokenize(text)]

model = Word2Vec(
    sentences,
    vector_size=100,   # embedding dimension
    window=5,          # context window
    min_count=1,       # keep every word (tiny corpus)
    sg=1,              # 1 = Skip-gram, 0 = CBOW — try both
    epochs=50,
)
print("vocab:", list(model.wv.key_to_index))

In [ ]:
vec = model.wv["learning"]
print("shape:", vec.shape)
print(model.wv.most_similar("learning"))  # meaningful only with a real corpus

## Pretrained GoogleNews vectors (lecture 27)

The course uses GoogleNews-vectors-negative300.bin (~1.6 GB). Download it out-of-band into `data/` (git-ignored) — e.g. from the gensim data page or Kaggle — then:

In [ ]:
# from gensim.models import KeyedVectors
# wv = KeyedVectors.load_word2vec_format("data/GoogleNews-vectors-negative300.bin", binary=True)
# print(wv.most_similar("king"))
# print(wv.most_similar(positive=["king", "woman"], negative=["man"]))  # famous analogy

## Average Word2Vec → features for any classifier

In [ ]:
import numpy as np

def avg_word2vec(sentence, wv):
    vectors = [wv[w] for w in sentence if w in wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(wv.vector_size)

doc_vectors = np.array([avg_word2vec(s, model.wv) for s in sentences])
print("document matrix:", doc_vectors.shape)  # (n_docs, vector_size) — feed to sklearn